In [39]:
import pandas as pd
import os
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [40]:
url = URL.create(
    drivername="mysql+mysqldb",
    username="root",
    password=os.getenv("MYSQL_PASSWORD"),
    host="127.0.0.1",
    port=3306,
    database="case_study"
)

engine = create_engine(url)

In [3]:
df = pd.DataFrame(pd.read_csv(r"C:\Users\allex\Downloads\zomato.csv"))
df.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


In [10]:
df_filtered = df[['name','rate','votes','location',	'rest_type','cuisines',	'approx_cost(for two people)']]
df_filtered.head()

,name,rate,votes,location,rest_type,cuisines,approx_cost(for two people)
0,Jalsa,4.1/5,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800
1,Spice Elephant,4.1/5,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800
2,San Churro Cafe,3.8/5,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800
3,Addhuri Udupi Bhojana,3.7/5,88,Banashankari,Quick Bites,"South Indian, North Indian",300
4,Grand Village,3.8/5,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600


In [20]:
df_filtered.info()

<class 'pandas.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   name                         51717 non-null  str    
 1   rate                         41665 non-null  float64
 2   votes                        51717 non-null  int64  
 3   location                     51696 non-null  str    
 4   rest_type                    51490 non-null  str    
 5   cuisines                     51672 non-null  str    
 6   approx_cost(for two people)  51371 non-null  str    
dtypes: float64(1), int64(1), str(5)
memory usage: 6.0 MB


In [18]:
# Step 1: Replace 'NEW' and '-' with NaN
df_filtered['rate'] = df_filtered['rate'].replace(['NEW', '-'], np.nan)

# Step 2: Extract numerator (eg. 4.5/5 = 4.5, 3/5 = 3.0)
df_filtered['rate'] = df_filtered['rate'].str.split('/').str[0]

# Step 3: Convert to float
df_filtered['rate'] = df_filtered['rate'].astype(float)

In [23]:
df_filtered['approx_cost(for two people)'] = pd.to_numeric(df_filtered['approx_cost(for two people)'],errors='coerce')

In [33]:
df_filtered.info()

<class 'pandas.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   name                         51717 non-null  str    
 1   rate                         41665 non-null  float64
 2   votes                        51717 non-null  int64  
 3   location                     51696 non-null  str    
 4   rest_type                    51490 non-null  str    
 5   cuisines                     51672 non-null  str    
 6   approx_cost(for two people)  44454 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 5.9 MB


In [38]:
df_filtered.isna().sum()

name                               0
rate                           10052
votes                              0
location                          21
rest_type                        227
cuisines                          45
approx_cost(for two people)     7263
dtype: int64

In [42]:
df_filtered.to_sql('zomato_bangalore_restaurant',con=engine,if_exists='replace',index=False,method='multi', chunksize=5000)

51717